In [1]:
# import os

# def generate_sh_ltf_script(dataset, model_type="LLM", parallel_num=3):
#     if model_type == 'sota':
#         script_content = f'''#!/bin/bash
# dir_path="../scripts/long_term_forecast/{dataset}_script"
# echo "Current directory: $(pwd)"
# find "$dir_path" -maxdepth 1 -name '*.sh' -print0 | xargs -0 -n 1 -P 1 bash
# echo "All scripts have been started."
#     '''        

#     elif model_type == 'non_Transformer':
#         script_content = f'''#!/bin/bash
# dir_path="./scripts/long_term_forecast/{dataset}_script/gym_{model_type}"
# echo "Current directory: $(pwd)"
# find "$dir_path" -name '*.sh' ! -name '*Transformer*.sh' -print0| shuf -z | xargs -0 -n 1 -P {parallel_num} bash
# echo "All scripts have been started."
#     '''
#     else:
#         script_content = f'''#!/bin/bash
# dir_path="./scripts/long_term_forecast/{dataset}_script/gym_{model_type}"
# echo "Current directory: $(pwd)"
# find "$dir_path" -name '*{model_type}*.sh' -print0 | shuf -z | xargs -0 -n 1 -P {parallel_num} bash
# echo "All scripts have been started."
#         '''
    
#     filename = f"../scripts/exp_{model_type}/exp_long_term_forecasting_gym_{dataset}.sh"
#     with open(filename, 'w') as f:
#         f.write(script_content)
#     print(f"生成成功 ➜ {filename}")


# # long term forecasting
# datasets = ['ETTh1','ECL', 'ETTh1', 'ETTh2', 'ETTm1', 'ETTm2', 'Exchange', 'ILI', 'Traffic', 'Weather','NYSE','NASDAQ']
# model_types = ['sota', 'non_Transformer', 'Transformer', 'LLM', 'TSFM']
# for model_type in model_types:
#     os.makedirs(f'./scripts/exp_{model_type}', exist_ok=True)

# configs = [
#     {
#         "model_type": model_type,
#         "dataset": dataset,  # 新增数据集字段
#         "parallel_num": 1 if dataset in ['ECL', 'Traffic'] else 5
#     }
#     for dataset in datasets
#     for model_type in model_types
# ]

# for config in configs:
#     generate_sh_ltf_script(**config)

In [ ]:
import os

def generate_sh_ltf_script(dataset, gym_type, source, parallel_num=3):
    """
    生成并行运行脚本
    :param dataset: 数据集名称 (e.g., ETTh1)
    :param gym_type: 架构类型 (MLP, GRU, Transformer)
    :param source: 来源 (sota, random)
    :param parallel_num: 并行进程数
    """
    # 1. 目标目录：所有脚本都在 gym_{gym_type} 下
    # 例如: ./scripts/long_term_forecast/ETTh1_script/gym_MLP
    target_dir_path = f"./scripts/long_term_forecast/{dataset}_script/gym_{gym_type}"
    
    # 2. 确定 ID 前缀匹配规则
    # 10开头为Random, 11开头为SOTA
    if source == 'random':
        file_pattern = "TSGym10*.sh"
    elif source == 'sota':
        file_pattern = "TSGym11*.sh"
    else:
        raise ValueError(f"Unknown source: {source}")

    script_content = f'''#!/bin/bash
# -------------------------------------------------------
# Auto-generated script for {dataset} | {gym_type} | {source}
# Target Folder: {target_dir_path}
# Pattern: {file_pattern}
# -------------------------------------------------------
dir_path="{target_dir_path}"

if [ ! -d "$dir_path" ]; then
  echo "Error: Directory $dir_path does not exist."
  exit 1
fi

echo "Searching in: $dir_path"
echo "Pattern: {file_pattern}"
echo "Starting execution with {parallel_num} parallel jobs..."

# 使用 -name 根据 ID 前缀进行筛选
find "$dir_path" -maxdepth 1 -name '{file_pattern}' -print0 | shuf -z | xargs -0 -n 1 -P {parallel_num} bash

echo "-------------------------------------------------------"
echo "All scripts for {gym_type} ({source}) have been triggered."
''' 
    
    # 生成的 runner 脚本存放位置，按 gym_type 分类
    # 例如: ../scripts/exp_MLP/run_ETTh1_sota.sh
    save_dir = f"../scripts/exp_{gym_type}"
    os.makedirs(save_dir, exist_ok=True)
    
    filename = f"{save_dir}/run_{dataset}_{source}.sh"
    
    with open(filename, 'w') as f:
        f.write(script_content)
    
    # 赋予执行权限
    try:
        os.chmod(filename, 0o755)
    except OSError:
        pass # Windows下可能忽略
        
    print(f"生成的Runner脚本 ➜ {filename}")

# ==========================================
# 配置与执行
# ==========================================

# 1. 定义数据集
datasets = ['ETTh1', 'ETTh2', 'ETTm1', 'ETTm2', 'ECL', 'Exchange', 'ILI', 'Traffic', 'Weather', 'NYSE', 'NASDAQ']

# 2. 定义架构和来源
# 注意：你现在的目录名就是 gym_MLP, gym_GRU, gym_Transformer，没有 non_Transformer 了
gym_types = ['MLP', 'GRU', 'Transformer'] # TODO: 可扩展为 LLM, TSFM
sources = ['sota', 'random']

# 3. 生成配置列表
configs = []
for dataset in datasets:
    # 针对大数据集降低并行度
    if dataset in ['ECL', 'Traffic']:
        p_num = 1 
    elif dataset in ['Weather', 'Exchange']:
        p_num = 3
    else:
        p_num = 5 

    for gym_type in gym_types:
        for source in sources:
            configs.append({
                "dataset": dataset,
                "gym_type": gym_type,
                "source": source,
                "parallel_num": p_num
            })

# 4. 执行生成
print(f"开始生成 {len(configs)} 个并行执行脚本...")
for config in configs:
    generate_sh_ltf_script(**config)

print("\n全部完成！请检查 scripts/exp_MLP/ 等目录。")

开始生成 66 个并行执行脚本...
生成的Runner脚本 ➜ ../scripts/exp_MLP/run_ETTh1_sota.sh
生成的Runner脚本 ➜ ../scripts/exp_MLP/run_ETTh1_random.sh
生成的Runner脚本 ➜ ../scripts/exp_GRU/run_ETTh1_sota.sh
生成的Runner脚本 ➜ ../scripts/exp_GRU/run_ETTh1_random.sh
生成的Runner脚本 ➜ ../scripts/exp_Transformer/run_ETTh1_sota.sh
生成的Runner脚本 ➜ ../scripts/exp_Transformer/run_ETTh1_random.sh
生成的Runner脚本 ➜ ../scripts/exp_MLP/run_ETTh2_sota.sh
生成的Runner脚本 ➜ ../scripts/exp_MLP/run_ETTh2_random.sh
生成的Runner脚本 ➜ ../scripts/exp_GRU/run_ETTh2_sota.sh
生成的Runner脚本 ➜ ../scripts/exp_GRU/run_ETTh2_random.sh
生成的Runner脚本 ➜ ../scripts/exp_Transformer/run_ETTh2_sota.sh
生成的Runner脚本 ➜ ../scripts/exp_Transformer/run_ETTh2_random.sh
生成的Runner脚本 ➜ ../scripts/exp_MLP/run_ETTm1_sota.sh
生成的Runner脚本 ➜ ../scripts/exp_MLP/run_ETTm1_random.sh
生成的Runner脚本 ➜ ../scripts/exp_GRU/run_ETTm1_sota.sh
生成的Runner脚本 ➜ ../scripts/exp_GRU/run_ETTm1_random.sh
生成的Runner脚本 ➜ ../scripts/exp_Transformer/run_ETTm1_sota.sh
生成的Runner脚本 ➜ ../scripts/exp_Transformer/run_ETTm1_random.

In [ ]:
# # long term forecasting
# datasets = ['ETTh1','ECL', 'ETTh1', 'ETTh2', 'ETTm1', 'ETTm2', 'Exchange', 'ILI', 'Traffic', 'Weather','NYSE','NASDAQ']
# model_types = ['sota', 'non_Transformer', 'Transformer', 'LLM', 'TSFM']
# for model_type in model_types:
#     os.makedirs(f'./scripts/exp_{model_type}', exist_ok=True)

# configs = [
#     {
#         "model_type": model_type,
#         "dataset": dataset,  # 新增数据集字段
#         "parallel_num": 1 if dataset in ['ECL', 'Traffic'] else 5
#     }
#     for dataset in datasets
#     for model_type in model_types
# ]

# for config in configs:
#     generate_sh_ltf_script(**config)

In [ ]:
# def generate_sh_stf_script(dataset, model_type="LLM", parallel_num=3):
#     if model_type == 'sota':
#         script_content = f'''#!/bin/bash
# dir_path="./scripts/short_term_forecast"
# echo "Current directory: $(pwd)"
# find "$dir_path" -maxdepth 1 -name '*.sh' -print0 | xargs -0 -n 1 -P 1 bash
# echo "All scripts have been started."
#     '''        

#     elif model_type == 'non_Transformer':
#         script_content = f'''#!/bin/bash
# dir_path="./scripts/short_term_forecast/gym_{model_type}"
# echo "Current directory: $(pwd)"
# find "$dir_path" -name '*.sh' ! -name '*Transformer*.sh' -print0| shuf -z | xargs -0 -n 1 -P {parallel_num} bash
# echo "All scripts have been started."
#     '''
#     else:
#         script_content = f'''#!/bin/bash
# dir_path="./scripts/short_term_forecast/gym_{model_type}"
# echo "Current directory: $(pwd)"
# find "$dir_path" -name '*{model_type}*.sh' -print0 | shuf -z | xargs -0 -n 1 -P {parallel_num} bash
# echo "All scripts have been started."
#         '''
    
#     filename = f"./scripts/exp_{model_type}/exp_short_term_forecasting_gym_{dataset}.sh"
#     with open(filename, 'w') as f:
#         f.write(script_content)
#     print(f"生成成功 ➜ {filename}")

In [ ]:
# # short term forecasting
# datasets = ['M4']
# model_types = ['sota', 'non_Transformer', 'Transformer', 'LLM', 'TSFM']
# for model_type in model_types:
#     os.makedirs(f'./scripts/exp_{model_type}', exist_ok=True)

# configs = [
#     {
#         "model_type": model_type,
#         "dataset": dataset,  # 新增数据集字段
#         "parallel_num": 10
#     }
#     for dataset in datasets
#     for model_type in model_types
# ]

# for config in configs:
#     generate_sh_stf_script(**config)

生成成功 ➜ ./scripts/exp_sota/exp_short_term_forecasting_gym_M4.sh
生成成功 ➜ ./scripts/exp_non_Transformer/exp_short_term_forecasting_gym_M4.sh
生成成功 ➜ ./scripts/exp_Transformer/exp_short_term_forecasting_gym_M4.sh
生成成功 ➜ ./scripts/exp_LLM/exp_short_term_forecasting_gym_M4.sh
生成成功 ➜ ./scripts/exp_TSFM/exp_short_term_forecasting_gym_M4.sh


In [ ]:
# # long term forecasting (add new datasets)
# datasets = ['covid-19', 'fred-md']
# model_types = ['sota', 'non_Transformer', 'Transformer', 'LLM', 'TSFM']
# for model_type in model_types:
#     os.makedirs(f'./scripts/exp_{model_type}', exist_ok=True)

# configs = [
#     {
#         "model_type": model_type,
#         "dataset": dataset,  # 新增数据集字段
#         "parallel_num": 5
#     }
#     for dataset in datasets
#     for model_type in model_types
# ]

# for config in configs:
#     generate_sh_ltf_script(**config)

In [ ]:
# # long term forecasting
# datasets = ['NYSE','NASDAQ']
# model_types = ['non_Transformer', 'Transformer', 'LLM', 'TSFM']
# for model_type in model_types:
#     os.makedirs(f'./scripts/exp_{model_type}', exist_ok=True)

# configs = [
#     {
#         "model_type": model_type,
#         "dataset": dataset,  # 新增数据集字段
#         "parallel_num": 1 if dataset in ['ECL', 'Traffic'] else 5
#     }
#     for dataset in datasets
#     for model_type in model_types
# ]

# for config in configs:
#     generate_sh_ltf_script(**config)

生成成功 ➜ ../scripts/exp_non_Transformer/exp_long_term_forecasting_gym_NYSE.sh
生成成功 ➜ ../scripts/exp_Transformer/exp_long_term_forecasting_gym_NYSE.sh
生成成功 ➜ ../scripts/exp_LLM/exp_long_term_forecasting_gym_NYSE.sh
生成成功 ➜ ../scripts/exp_TSFM/exp_long_term_forecasting_gym_NYSE.sh
生成成功 ➜ ../scripts/exp_non_Transformer/exp_long_term_forecasting_gym_NASDAQ.sh
生成成功 ➜ ../scripts/exp_Transformer/exp_long_term_forecasting_gym_NASDAQ.sh
生成成功 ➜ ../scripts/exp_LLM/exp_long_term_forecasting_gym_NASDAQ.sh
生成成功 ➜ ../scripts/exp_TSFM/exp_long_term_forecasting_gym_NASDAQ.sh
